In [1]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path


def find_repository_root(start=Path.cwd()):
    """Find the repository root from Jupyter's current working directory."""
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from inside "
        "NYC_Healthcare_Accessibility."
    )


REPO_ROOT = find_repository_root()
INPUT_DIR = REPO_ROOT / "data" / "processed" / "intermediate" / "ewm_inputs"
OUTPUT_DIR = REPO_ROOT / "data" / "processed" / "intermediate" / "ewm_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Libraries loaded.")

Libraries loaded.


In [3]:

df = pd.read_csv(INPUT_DIR / "The Bronx! - The_Bronx_Indicators_6.csv")
df.head(342)

,Unnamed: 0,from_id,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,1,o_360050002001,5569.749367,1358.969,2,40.150000,23.116667,12.050000
1,2,o_360050002002,5230.435620,2268.864,1,53.933333,38.266667,5.200000
2,3,o_360050002003,6012.862969,1130.742,2,39.750000,19.166667,12.450000
3,4,o_360050004001,6640.363689,754.017,1,31.900000,12.766667,7.116667
4,5,o_360050004002,5519.310367,1308.530,2,39.183333,22.150000,13.016667
...,...,...,...,...,...,...,...,...
337,338,o_360050175002,1721.253988,1523.379,0,27.366667,25.783333,2.233333
338,339,o_360050175003,1421.893303,693.378,0,17.833333,11.683333,1.133333
339,340,o_360050175004,1224.514164,1014.669,0,18.466667,17.216667,1.150000
340,341,o_360050175005,1245.444094,773.121,0,17.083333,13.083333,1.883333


In [4]:
print(df.shape)
print(df.columns.tolist())

(1156, 8)
['Unnamed: 0', 'from_id', 'total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']


In [5]:
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

df["GEOID_TEXT"] = df["from_id"].astype(str).str.replace("o_", "", regex=False)

df[["from_id", "GEOID_TEXT"]].head(343)

,from_id,GEOID_TEXT
0,o_360050002001,360050002001
1,o_360050002002,360050002002
2,o_360050002003,360050002003
3,o_360050004001,360050004001
4,o_360050004002,360050004002
...,...,...
338,o_360050175003,360050175003
339,o_360050175004,360050175004
340,o_360050175005,360050175005
341,o_360050175006,360050175006


In [7]:
benefit_cols = []
cost_cols = ["total_distance",

    "walking_distance",

    "transfers",

    "travel_time_total",

    "walking_time",

    "wait_time_total"]

criteria_cols = benefit_cols + cost_cols

print("Benefit indicators, higher is better:")
print(benefit_cols)

print("\nCost indicators, lower is better:")
print(cost_cols)

print("\nAll criteria:")
print(criteria_cols)

Benefit indicators, higher is better:
[]

Cost indicators, lower is better:
['total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']

All criteria:
['total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']


In [9]:
min_max_table = pd.DataFrame({
    "min": df[criteria_cols].min(),
    "max": df[criteria_cols].max()
})

print("Min and max for each indicator:")
display(min_max_table)

Min and max for each indicator:


,min,max
total_distance,469.889553,12386.902260
walking_distance,166.283000,3018.720000
transfers,0.000000,2.000000
travel_time_total,1.000000,64.183333
walking_time,2.800000,50.716667
wait_time_total,1.016667,15.966667


In [10]:
# All of our current indicators are cost indicators
# Lower = better, so we use: (max - value) / (max - min)

normalized = pd.DataFrame(index=df.index)

for col in criteria_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    
    normalized[col] = (max_val - df[col]) / (max_val - min_val)

print("Normalized values:")
display(normalized.head())

Normalized values:


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.572052,0.581871,0.0,0.380375,0.576000,0.261984
1,0.600525,0.262883,0.5,0.162226,0.259826,0.720178
2,0.534869,0.661882,0.0,0.386705,0.658435,0.235229
3,0.482213,0.793954,0.5,0.510947,0.792000,0.591973
4,0.576285,0.599554,0.0,0.395674,0.596174,0.197324


In [11]:
r_column_sums = normalized[criteria_cols].sum()

print("Step 2 preparation: Sum of each standardized column")
print("These sums go in the denominator for p_ij.")
display(r_column_sums)

Step 2 preparation: Sum of each standardized column
These sums go in the denominator for p_ij.


total_distance        969.579129
walking_distance      744.199366
transfers            1042.500000
travel_time_total     711.217884
walking_time          740.634435
wait_time_total       958.792642
dtype: float64

In [12]:
P = normalized[criteria_cols] / r_column_sums

print("Step 2: Probability matrix p_ij")
print("Each standardized value is divided by its column total.")
display(P.head())

Step 2: Probability matrix p_ij
Each standardized value is divided by its column total.


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.000590,0.000782,0.00000,0.000535,0.000778,0.000273
1,0.000619,0.000353,0.00048,0.000228,0.000351,0.000751
2,0.000552,0.000889,0.00000,0.000544,0.000889,0.000245
3,0.000497,0.001067,0.00048,0.000718,0.001069,0.000617
4,0.000594,0.000806,0.00000,0.000556,0.000805,0.000206


In [13]:
print("values should add up to one for each column, since they are probabilities.")
display(P.sum())

values should add up to one for each column, since they are probabilities.


total_distance       1.0
walking_distance     1.0
transfers            1.0
travel_time_total    1.0
walking_time         1.0
wait_time_total      1.0
dtype: float64

In [14]:
P_safe = P.replace(0, 1e-12)

print("Step 3 preparation: Replace 0 values so ln(0) does not break")
display(P_safe.head())

Step 3 preparation: Replace 0 values so ln(0) does not break


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.000590,0.000782,1.000000e-12,0.000535,0.000778,0.000273
1,0.000619,0.000353,4.796163e-04,0.000228,0.000351,0.000751
2,0.000552,0.000889,1.000000e-12,0.000544,0.000889,0.000245
3,0.000497,0.001067,4.796163e-04,0.000718,0.001069,0.000617
4,0.000594,0.000806,1.000000e-12,0.000556,0.000805,0.000206


In [15]:
ln_P = np.log(P_safe)

print("Step 3: Natural log of p_ij")
display(ln_P.head())

Step 3: Natural log of p_ij


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,-7.435387,-7.153815,-27.631021,-7.533578,-7.159155,-8.205145
1,-7.386813,-7.948357,-7.642524,-8.385742,-7.955250,-7.193931
2,-7.502596,-7.024976,-27.631021,-7.517071,-7.025397,-8.312873
3,-7.606231,-6.843039,-7.642524,-7.238468,-6.840701,-7.389969
4,-7.428016,-7.123878,-27.631021,-7.494144,-7.124730,-8.488581


In [16]:
P_ln_P = P_safe * ln_P

print("Step 4: p_ij times ln(p_ij)")
display(P_ln_P.head(8))

Step 4: p_ij times ln(p_ij)


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,-0.004387,-0.005593,-2.763102e-11,-0.004029,-0.005568,-0.002242
1,-0.004575,-0.002808,-3.665479e-03,-0.001913,-0.002791,-0.005404
2,-0.004139,-0.006248,-2.763102e-11,-0.004087,-0.006246,-0.002039
3,-0.003783,-0.007301,-3.665479e-03,-0.005200,-0.007315,-0.004563
4,-0.004415,-0.005739,-2.763102e-11,-0.004169,-0.005735,-0.001747
5,-0.004083,-0.006799,-2.763102e-11,-0.004332,-0.006796,-0.001137
6,-0.005131,-0.005755,-3.665479e-03,-0.004487,-0.005767,-0.003217
7,-0.005104,-0.006589,-3.665479e-03,-0.005049,-0.006586,-0.005252


In [17]:
p_ln_p_sums = P_ln_P.sum(axis=0)

print("Step 4 preparation: Sum of p_ij * ln(p_ij) for each indicator")
display(p_ln_p_sums)

Step 4 preparation: Sum of p_ij * ln(p_ij) for each indicator


total_distance      -7.039920
walking_distance    -7.011616
transfers           -7.016198
travel_time_total   -7.015702
walking_time        -7.010649
wait_time_total     -7.033722
dtype: float64

In [18]:
# Step 5: Calculate entropy for each indicator

n = len(normalized)
k = 1 / np.log(n)

entropy = -k * p_ln_p_sums

print("Number of rows:", n)
print("k value:", k)
print("Step 5: Entropy for each indicator")
display(entropy)

Number of rows: 1156
k value: 0.1417892460256667
Step 5: Entropy for each indicator


total_distance       0.998185
walking_distance     0.994172
transfers            0.994821
travel_time_total    0.994751
walking_time         0.994035
wait_time_total      0.997306
dtype: float64

In [44]:
# df["fare"].value_counts()

In [19]:
# Step 6: Diversity
# Diversity tells us how much useful variation each indicator has

diversity = 1 - entropy

print("Step 6: Diversity for each indicator")
display(diversity)

Step 6: Diversity for each indicator


total_distance       0.001815
walking_distance     0.005828
transfers            0.005179
travel_time_total    0.005249
walking_time         0.005965
wait_time_total      0.002694
dtype: float64

In [20]:
# Step 7: Calculate entropy weights
# Weight = diversity of one indicator / total diversity of all indicators

weights = diversity / diversity.sum()

print("Step 7: Entropy weights for each indicator")
display(weights)

Step 7: Entropy weights for each indicator


total_distance       0.067901
walking_distance     0.218043
transfers            0.193735
travel_time_total    0.196367
walking_time         0.223172
wait_time_total      0.100782
dtype: float64

In [21]:
weights_table = pd.DataFrame({
    "entropy": entropy,
    "diversity": diversity,
    "weight": weights
})

print("Final entropy weight table:")
display(weights_table)

Final entropy weight table:


,entropy,diversity,weight
total_distance,0.998185,0.001815,0.067901
walking_distance,0.994172,0.005828,0.218043
transfers,0.994821,0.005179,0.193735
travel_time_total,0.994751,0.005249,0.196367
walking_time,0.994035,0.005965,0.223172
wait_time_total,0.997306,0.002694,0.100782


In [22]:
display(weights_table.sort_values(by="weight", ascending=False))

,entropy,diversity,weight
walking_time,0.994035,0.005965,0.223172
walking_distance,0.994172,0.005828,0.218043
travel_time_total,0.994751,0.005249,0.196367
transfers,0.994821,0.005179,0.193735
wait_time_total,0.997306,0.002694,0.100782
total_distance,0.998185,0.001815,0.067901


In [23]:
# Step 8: Calculate final EWM accessibility score
# Formula: score for each block group = sum(normalized value * indicator weight)

df["ewm_accessibility_score"] = (normalized[criteria_cols] * weights).sum(axis=1)

print("Step 8: Final EWM accessibility score")
display(df[["from_id", "GEOID_TEXT", "ewm_accessibility_score"]].head(30))

Step 8: Final EWM accessibility score


,from_id,GEOID_TEXT,ewm_accessibility_score
0,o_360050002001,360050002001,0.395359
1,o_360050002002,360050002002,0.357386
2,o_360050002003,360050002003,0.427224
3,o_360050004001,360050004001,0.639472
4,o_360050004002,360050004002,0.400492
5,o_360050004003,360050004003,0.450674
6,o_360050004004,360050004004,0.532952
7,o_360050016001,360050016001,0.620220
8,o_360050016002,360050016002,0.451525
9,o_360050016003,360050016003,0.553612


In [24]:
print("Score summary:")
display(df["ewm_accessibility_score"].describe())

Score summary:


count    1156.000000
mean        0.719420
std         0.124000
min         0.100785
25%         0.636427
50%         0.735084
75%         0.810827
max         0.970741
Name: ewm_accessibility_score, dtype: float64

In [25]:
final_results = df[
    ["GEOID_TEXT", "from_id"] + criteria_cols + ["ewm_accessibility_score"]
].copy()

display(final_results.head(20))

,GEOID_TEXT,from_id,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total,ewm_accessibility_score
0,360050002001,o_360050002001,5569.749367,1358.969,2,40.150000,23.116667,12.050000,0.395359
1,360050002002,o_360050002002,5230.435620,2268.864,1,53.933333,38.266667,5.200000,0.357386
2,360050002003,o_360050002003,6012.862969,1130.742,2,39.750000,19.166667,12.450000,0.427224
3,360050004001,o_360050004001,6640.363689,754.017,1,31.900000,12.766667,7.116667,0.639472
4,360050004002,o_360050004002,5519.310367,1308.530,2,39.183333,22.150000,13.016667,0.400492
5,360050004003,o_360050004003,6111.243578,934.865,2,38.050000,15.900000,14.150000,0.450674
6,360050004004,o_360050004004,4214.603700,1303.090,1,36.966667,21.966667,10.050000,0.532952
7,360050016001,o_360050016001,4264.470655,1010.007,1,32.983333,17.150000,5.550000,0.620220
8,360050016002,o_360050016002,4408.614288,1835.849,1,40.833333,31.000000,7.900000,0.451525
9,360050016003,o_360050016003,3698.995871,1331.044,1,34.966667,22.583333,7.600000,0.553612


In [26]:
final_results.to_csv(OUTPUT_DIR / "bronx_ewm_results.csv", index=False)
weights_table.to_csv(OUTPUT_DIR / "bronx_ewm_weights.csv", index=True)

print("Saved bronx_ewm_results.csv")
print("Saved bronx_ewm_weights.csv")

Saved bronx_ewm_results.csv
Saved bronx_ewm_weights.csv
